In [94]:
import pandas as pd
from scipy import stats
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as st
import pingouin as pg


In [ ]:
childMeasuresPath = '/vol/tensusers2/wharmsen/edLing-RDR/automaticFluency/child/automatic_measures.csv'
child_egemaps = '/vol/tensusers2/wharmsen/edLing-RDR/automaticFluency/child/eGeMAPSv02_Functionals_88feat.tsv'

adultMeasuresPath = '/vol/tensusers2/wharmsen/edLing-RDR/automaticFluency/adult/automatic_measures.csv'
adult_egemaps = '/vol/tensusers2/wharmsen/edLing-RDR/automaticFluency/adult/eGeMAPSv02_Functionals_88feat.tsv'

splitOnSubjectiveFluencyDF = 'mdfsOneScoreDF_metadata.tsv'

In [96]:
childMeasures_raw = pd.read_csv(childMeasuresPath, index_col=[0])
childMeasuresEGeMAPS = pd.read_csv(child_egemaps, sep='\t').set_index('file')['F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2']
childMeasures = pd.concat([childMeasures_raw, childMeasuresEGeMAPS], axis=1)

childMeasures

,meanDurInterSilentPauses,stdDurInterSilentPauses,nrIntraSilentPausesPerWord,meanDurationIntraSilentPauses,stdDurationIntraSilentPauses,SpeechRate(nrSyllPerMinute),ArtRate(nrSyllPerMinute),wcpm,perc_correct,age,speakerID,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2
fn000561,1.162,1.132,0.225490,0.644,0.829,182.744275,268.652451,122.477546,0.994737,12.0,N000215,3.879719
fn000558,1.085,1.228,0.488095,0.401,0.676,146.383117,219.965987,86.610011,0.950893,11.0,N000214,2.734699
fn000555,1.277,1.501,0.349515,0.635,1.003,127.882166,199.700327,83.052624,0.988764,7.0,N000213,3.061695
fn000552,1.503,1.837,0.494681,0.530,1.077,109.810934,170.738291,75.956180,0.983146,6.0,N000212,3.029011
fn000549,1.940,2.069,0.829457,0.622,1.195,70.557162,145.405845,55.238831,0.967480,7.0,N000211,2.444679
...,...,...,...,...,...,...,...,...,...,...,...,...
fn000064,0.882,1.085,0.316456,0.415,0.683,173.354353,241.169673,106.179541,0.960784,9.0,N000030,4.735878
fn000062,1.125,1.211,0.314433,0.429,0.728,150.390989,199.056709,103.814675,0.973684,10.0,N000029,2.879627
fn000060,0.814,1.092,0.350785,0.469,0.750,138.807413,195.026313,96.394037,0.983146,10.0,N000028,4.512783
fn000051,1.852,1.573,0.569307,0.799,1.146,87.353131,185.241681,54.897818,0.963636,8.0,N000026,3.042332


In [97]:
# Read all adult measures
adultMeasures_raw = pd.read_csv(adultMeasuresPath, index_col=[0])
adultMeasuresEGeMAPS = pd.read_csv(adult_egemaps, sep='\t').set_index('file')['F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2']
adultMeasures = pd.concat([adultMeasures_raw, adultMeasuresEGeMAPS], axis=1)

### Add/remove missing values ### 

# 1.
adultMeasures['wcpm'] = adultMeasures['SpeechRate(nrWordsPerMinute)']

# 2. 
adultMeasures['perc_correct'] = [1.0] * len(adultMeasures)

# 3. Print and drop variables (measures) that have more than 27 files with missing values
columnsWithMissingValues = adultMeasures.isna().sum() > 27 # There are 27 files with missing pitch values
columnsWithMissingValues = columnsWithMissingValues.index[columnsWithMissingValues]
print(columnsWithMissingValues)

adultMeasures = adultMeasures.drop(list(columnsWithMissingValues), axis=1)

# Drop 27 rows with missing pitch values
adultMeasures = adultMeasures.dropna()

### Select for each adult speaker only one file ###

# Get unique speakers
uniqueSpeakers = set(adultMeasures['speakerID'])
print('nrUniqueSpeakers:', len(uniqueSpeakers))

# Select for each unique speaker only one file
filesToSelect = []
for speaker in uniqueSpeakers:
    firstOccurence = adultMeasures[adultMeasures['speakerID'] == speaker].index[0]
    filesToSelect.append(firstOccurence)

adultMeasures319 = adultMeasures.loc[filesToSelect]
adultMeasures319

Index([], dtype='object')
nrUniqueSpeakers: 324


,meanDurInterSilentPauses,stdDurInterSilentPauses,nrIntraSilentPausesPerWord,meanDurationIntraSilentPauses,stdDurationIntraSilentPauses,SpeechRate(nrSyllPerMinute),ArtRate(nrSyllPerMinute),speakerID,SpeechRate(nrWordsPerMinute),F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,wcpm,perc_correct
fn001133,0.953,0.376,0.147014,0.600,0.453,242.958450,319.501757,N00578,163.054335,5.382141,163.054335,1.0
fn001417,0.865,0.499,0.165719,0.485,0.467,247.927694,312.166803,N00516,153.675222,4.895164,153.675222,1.0
fn001324,1.146,0.678,0.181380,0.630,0.587,212.917473,297.719276,N00689,149.546320,7.619152,149.546320,1.0
fn001197,0.987,0.301,0.231207,0.478,0.402,211.405309,281.891475,N00697,135.762517,4.238081,135.762517,1.0
fn001166,1.332,0.540,0.140043,1.041,0.641,227.633565,369.542250,N00666,158.039125,5.530159,158.039125,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
fn001111,1.374,0.506,0.140600,0.719,0.607,253.529094,349.269698,N00567,162.762593,5.006020,162.762593,1.0
fn001174,1.175,0.608,0.128413,0.760,0.642,250.571911,339.059997,N00674,160.554337,5.628000,160.554337,1.0
fn001140,0.989,0.498,0.285536,0.424,0.436,201.042634,278.237356,N00640,137.456259,5.789406,137.456259,1.0
fn001304,1.534,1.484,0.240166,0.574,0.964,208.892130,287.906235,N00680,119.402247,4.820181,119.402247,1.0


Data matrix preperation

1. Add  group variables

In [98]:
adultMeasures319.index

Index(['fn001133', 'fn001417', 'fn001324', 'fn001197', 'fn001166', 'fn001052',
       'fn001062', 'fn001209', 'fn001504', 'fn001560',
       ...
       'fn001384', 'fn001162', 'fn001155', 'fn001356', 'fn001130', 'fn001111',
       'fn001174', 'fn001140', 'fn001304', 'fn001322'],
      dtype='object', length=324)

In [ ]:
# Split children in three groups
fluent35DF = pd.read_csv(splitOnSubjectiveFluencyDF).set_index('bestand')
child4_readers = fluent35DF[fluent35DF['finalScore_round'] == 4].index
child3_readers = fluent35DF[fluent35DF['finalScore_round'] == 3].index
child2_readers = fluent35DF[fluent35DF['finalScore_round'] == 2].index
print(child4_readers)

Index(['fn000123', 'fn000110', 'fn000139', 'fn000068', 'fn000090', 'fn000135',
       'fn000087', 'fn000092', 'fn000561', 'fn000104', 'fn000515', 'fn000049',
       'fn000155', 'fn000537', 'fn000082', 'fn000491', 'fn000510', 'fn000129',
       'fn000106', 'fn000098', 'fn000507', 'fn000070', 'fn000100', 'fn000096',
       'fn000157', 'fn000133', 'fn000121', 'fn000137', 'fn000499', 'fn000084',
       'fn000060', 'fn000534', 'fn000094', 'fn000115', 'fn000127'],
      dtype='object', name='bestand')


In [100]:
childMeasures.loc[child2_readers,'age'].mean()

8.0

In [101]:
# Remove children with fluency = 2
print(len(childMeasures))
childMeasures = childMeasures.drop(child2_readers)
print(len(childMeasures))

71
66


In [102]:
# Option 1: moderately fluent children, fluent children, adults
childMeasures['group_three'] = [0 if x in child3_readers else 1 for x in childMeasures.index]
adultMeasures319['group_three'] = [2 for x in range(len(adultMeasures319))]

In [103]:
childMeasures[childMeasures['group_three'] == 1].loc[:, 'age'].mean()

10.529411764705882

In [104]:
childMeasures['group_three'].value_counts()

group_three
1    35
0    31
Name: count, dtype: int64

2. Select the same columns for both dataframes

In [105]:
childMeasuresInAdultMeasures = [x for x in childMeasures.columns if x in adultMeasures319.columns]
childMeasuresNotInAdultMeasures = [x for x in childMeasures.columns if x not in adultMeasures319.columns]

adultMeasuresInChildMeasures = [x for x in adultMeasures319.columns if x in childMeasures.columns]
adultMeasuresNotInChildMeasures = [x for x in adultMeasures319.columns if x not in childMeasures.columns]

print("childMeasuresInAdultMeasures", childMeasuresInAdultMeasures)
print("childMeasuresNotInAdultMeasures", childMeasuresNotInAdultMeasures)
print("adultMeasuresInChildMeasures", adultMeasuresInChildMeasures)
print("adultMeasuresNotInChildMeasures", adultMeasuresNotInChildMeasures)

childMeasuresInAdultMeasures ['meanDurInterSilentPauses', 'stdDurInterSilentPauses', 'nrIntraSilentPausesPerWord', 'meanDurationIntraSilentPauses', 'stdDurationIntraSilentPauses', 'SpeechRate(nrSyllPerMinute)', 'ArtRate(nrSyllPerMinute)', 'wcpm', 'perc_correct', 'speakerID', 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2', 'group_three']
childMeasuresNotInAdultMeasures ['age']
adultMeasuresInChildMeasures ['meanDurInterSilentPauses', 'stdDurInterSilentPauses', 'nrIntraSilentPausesPerWord', 'meanDurationIntraSilentPauses', 'stdDurationIntraSilentPauses', 'SpeechRate(nrSyllPerMinute)', 'ArtRate(nrSyllPerMinute)', 'speakerID', 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2', 'wcpm', 'perc_correct', 'group_three']
adultMeasuresNotInChildMeasures ['SpeechRate(nrWordsPerMinute)']


In [106]:
measure_selection = [
 'meanDurInterSilentPauses',
 'stdDurInterSilentPauses',
 'nrIntraSilentPausesPerWord',
 'meanDurationIntraSilentPauses',
 'stdDurationIntraSilentPauses',
 'SpeechRate(nrSyllPerMinute)',
 'ArtRate(nrSyllPerMinute)',
 'F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2',
 'wcpm',
 'perc_correct',
 'group_three',
 ]


In [107]:
child_adult_DF = pd.concat([adultMeasures319, childMeasures]).dropna(axis=1)
child_adult_DF = child_adult_DF.loc[:, measure_selection]
child_adult_DF

,meanDurInterSilentPauses,stdDurInterSilentPauses,nrIntraSilentPausesPerWord,meanDurationIntraSilentPauses,stdDurationIntraSilentPauses,SpeechRate(nrSyllPerMinute),ArtRate(nrSyllPerMinute),F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,wcpm,perc_correct,group_three
fn001133,0.953,0.376,0.147014,0.600,0.453,242.958450,319.501757,5.382141,163.054335,1.000000,2
fn001417,0.865,0.499,0.165719,0.485,0.467,247.927694,312.166803,4.895164,153.675222,1.000000,2
fn001324,1.146,0.678,0.181380,0.630,0.587,212.917473,297.719276,7.619152,149.546320,1.000000,2
fn001197,0.987,0.301,0.231207,0.478,0.402,211.405309,281.891475,4.238081,135.762517,1.000000,2
fn001166,1.332,0.540,0.140043,1.041,0.641,227.633565,369.542250,5.530159,158.039125,1.000000,2
...,...,...,...,...,...,...,...,...,...,...,...
fn000066,1.164,1.193,0.531250,0.479,0.762,98.128818,142.925202,3.796093,65.804031,0.960674,0
fn000064,0.882,1.085,0.316456,0.415,0.683,173.354353,241.169673,4.735878,106.179541,0.960784,0
fn000062,1.125,1.211,0.314433,0.429,0.728,150.390989,199.056709,2.879627,103.814675,0.973684,0
fn000060,0.814,1.092,0.350785,0.469,0.750,138.807413,195.026313,4.512783,96.394037,0.983146,1


In [108]:
child_adult_DF['group_three'].value_counts()

group_three
2    324
1     35
0     31
Name: count, dtype: int64

# ANOVA tests

In [109]:
child_adult_DF[child_adult_DF['group_three'] == 2].std().round(2)

meanDurInterSilentPauses                     0.26
stdDurInterSilentPauses                      0.17
nrIntraSilentPausesPerWord                   0.04
meanDurationIntraSilentPauses                0.14
stdDurationIntraSilentPauses                 0.13
SpeechRate(nrSyllPerMinute)                 24.13
ArtRate(nrSyllPerMinute)                    24.04
F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2     1.37
wcpm                                        15.89
perc_correct                                 0.00
group_three                                  0.00
dtype: float64

In [110]:

# df has columns: "value" (numeric), "group" (categorical)
dfList = []
for var in measure_selection:
    welch = pg.welch_anova(dv=var, between='group_three', data=child_adult_DF)
    welch['var'] = [var] * len(welch)
    dfList.append(welch.loc[:, ['var', 'F', 'p-unc']].round(3))

pd.concat(dfList)

/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/parametric.py:1346: RuntimeWarning: invalid value encountered in scalar divide
  adj_grandmean = (weights * grp.mean(numeric_only=True)).sum() / weights.sum()
/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/parametric.py:1346: RuntimeWarning: invalid value encountered in scalar divide
  adj_grandmean = (weights * grp.mean(numeric_only=True)).sum() / weights.sum()
/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/parametric.py:1358: RuntimeWarning: divide by zero encountered in scalar divide
  ddof2 = 1 / lamb


,var,F,p-unc
0,meanDurInterSilentPauses,3.209,0.049
0,stdDurInterSilentPauses,151.586,0.000
0,nrIntraSilentPausesPerWord,41.099,0.000
0,meanDurationIntraSilentPauses,0.300,0.742
0,stdDurationIntraSilentPauses,90.914,0.000
0,SpeechRate(nrSyllPerMinute),127.457,0.000
0,ArtRate(nrSyllPerMinute),89.562,0.000
0,F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2,198.434,0.000
0,wcpm,114.733,0.000
0,perc_correct,0.000,1.000


## Games Howell post test

In [111]:
dfList = []

for var in measure_selection:
    gameshowellDF = pg.pairwise_gameshowell(data=child_adult_DF, dv=var,between='group_three', effsize='cohen').round(3)

    gameshowellDF['var'] = [var] * len(gameshowellDF)
    dfList.append(gameshowellDF)

pd.concat(dfList)

/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/pairwise.py:1058: RuntimeWarning: divide by zero encountered in divide
  tval = mn / np.sqrt(gvars[g1] / n[g1] + gvars[g2] / n[g2])
/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/pairwise.py:1059: RuntimeWarning: invalid value encountered in divide
  df = (gvars[g1] / n[g1] + gvars[g2] / n[g2]) ** 2 / (
/vol/tensusers5/wharmsen/virenv-wav2vec2/lib/python3.10/site-packages/pingouin/effsize.py:803: RuntimeWarning: divide by zero encountered in scalar divide
  d = (x.mean() - y.mean()) / poolsd


,A,B,mean(A),mean(B),diff,se,T,df,pval,cohen,var
0,0,1,1.224,1.167,0.057,0.068,0.835,63.211,0.683,0.206,meanDurInterSilentPauses
1,0,2,1.224,1.104,0.119,0.052,2.314,35.453,0.067,0.451,meanDurInterSilentPauses
2,1,2,1.167,1.104,0.062,0.049,1.263,40.838,0.424,0.235,meanDurInterSilentPauses
0,0,1,1.308,1.131,0.177,0.082,2.147,62.519,0.089,0.530,stdDurInterSilentPauses
1,0,2,1.308,0.479,0.828,0.062,13.462,31.401,0.000,4.422,stdDurInterSilentPauses
2,1,2,1.131,0.479,0.651,0.056,11.576,35.909,0.000,3.466,stdDurInterSilentPauses
0,0,1,0.360,0.242,0.117,0.021,5.674,48.571,0.000,1.439,nrIntraSilentPausesPerWord
1,0,2,0.360,0.205,0.154,0.018,8.539,31.005,0.000,3.130,nrIntraSilentPausesPerWord
2,1,2,0.242,0.205,0.037,0.011,3.478,37.468,0.004,0.841,nrIntraSilentPausesPerWord
0,0,1,0.599,0.616,-0.017,0.032,-0.519,61.146,0.862,-0.129,meanDurationIntraSilentPauses
